# Filler detector, part 2: test on the pilot students (C4 PP2)

Checks the part 1 model on the whole recordings of P01 to P10 and compares what it finds with the
fillers CrisperWhisper marked. Then it teaches the model the students' voices, testing each
student with a model that never heard that student (leave-one-student-out), and saves the final
model `filler_cnn_v2.pt` to Drive.

**Before you start:** Runtime, Change runtime type, **T4 GPU**, Save. Part 1 must have run
(Drive folder `c4-filler` has `filler_cnn_v1.pt`).

**Steps:** Runtime, Run all. Allow Google Drive. When cell 2 shows **Choose files**, open
`Desktop/Research Project/Research Data/Recordings` and select these 20 files together (hold
Ctrl and click): `P01_student.m4a` to `P10_student.m4a` and `P01_student.words.json` to
`P10_student.words.json`. Not P00 (practice), not the examiner files, and not the P04 files
directly inside `Research Data` (another person). About 15 to 30 minutes in total.

**Privacy:** the recordings go only into this Colab session, never into Drive. At the end,
Runtime, Disconnect and delete runtime removes them. Drive keeps only the model and the counts.

In [ ]:
# 1. Libraries, Google Drive and the part 1 model
!pip -q install librosa
import copy
import datetime
import hashlib
import json
import subprocess
from pathlib import Path
from zoneinfo import ZoneInfo

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from google.colab import drive, files

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/c4-filler")
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)


def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))


class FillerCNN(nn.Module):
    """The same network as part 1: 1 second of 16 kHz audio in, a score for each label out."""

    def __init__(self):
        super().__init__()
        mel = librosa.filters.mel(sr=16000, n_fft=400, n_mels=64)
        self.register_buffer("mel", torch.tensor(mel, dtype=torch.float32))
        self.register_buffer("window", torch.hann_window(400))
        self.norm = nn.BatchNorm2d(1)
        self.net = nn.Sequential(
            block(1, 32), block(32, 64), block(64, 128), block(128, 128),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 6),
        )

    def forward(self, wav):  # wav: (batch, 16000) floats in [-1, 1]
        power = torch.stft(wav, 400, 160, window=self.window, return_complex=True).abs() ** 2
        logmel = torch.log(self.mel @ power + 1e-6).unsqueeze(1)
        return self.net(self.norm(logmel))


ckpt = torch.load(DRIVE / "filler_cnn_v1.pt", map_location=dev)
LABELS, FILLERS = ckpt["labels"], ckpt["fillers"]
v1 = FillerCNN().to(dev)
v1.load_state_dict(ckpt["state_dict"])
v1.eval()
print("Part 1 model loaded:", LABELS)

In [ ]:
# 2. Upload the 20 files (P01 to P10: *_student.m4a and *_student.words.json)
uploaded = files.upload()
codes = sorted(n[:3] for n in uploaded if n.endswith("_student.m4a") and not n.startswith("P00"))
assert codes, "No *_student.m4a files were uploaded."
assert all(f"{c}_student.words.json" in uploaded for c in codes), "Each .m4a needs its .words.json"
print(len(codes), "students:", codes)

SR = 16000
TARGET_RMS = 0.1  # podcast-like loudness for the speech parts (Zoom recordings are quieter)
students = {}
for c in codes:
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", f"{c}_student.m4a", "-ac", "1", "-ar", str(SR), f"{c}.wav"],
        check=True,
    )
    audio, _ = librosa.load(f"{c}.wav", sr=SR)
    speech = librosa.effects.split(audio, top_db=40) / SR  # (start, end) in seconds of the non-silent parts
    loud = np.concatenate([audio[int(a * SR) : int(b * SR)] for a, b in speech])
    audio = np.clip(audio * TARGET_RMS / np.sqrt(np.mean(loud**2)), -1, 1).astype(np.float32)
    words = json.load(open(f"{c}_student.words.json", encoding="utf-8"))["words"]
    is_filler = [w["word"].strip().upper() in ("[UM]", "[UH]") for w in words]
    students[c] = {
        "audio": audio,
        "speech": speech,
        "fillers": np.array([(w["start"] + w["end"]) / 2 for w, f in zip(words, is_filler) if f]),
        "filler_tags": [w["word"].strip().upper() for w, f in zip(words, is_filler) if f],
        "words": np.array([(w["start"] + w["end"]) / 2 for w, f in zip(words, is_filler) if not f]),
    }
    print(f"{c}: {len(audio) / SR / 60:.1f} min, {sum(is_filler)} CrisperWhisper fillers")

In [ ]:
# 3. How fillers are found in a whole recording and compared with CrisperWhisper
# These settings were fixed before looking at any results.
SR, HOP = 16000, 0.1  # a 1-second window centred every 0.1 s
THRESHOLD = 0.5  # uh + um score needed to count a window as a filler
MIN_WINDOWS = 2  # a filler needs at least 2 windows in a row
MATCH_S = 0.5  # a found filler matches a CrisperWhisper filler within 0.5 s


def windows(audio, centres):
    """1-second pieces of audio centred on each time (seconds); silence beyond the ends."""
    padded = np.pad(audio, SR // 2)
    starts = np.clip((np.asarray(centres) * SR).astype(int), 0, len(audio))
    return np.stack([padded[s : s + SR] for s in starts]).astype(np.float32)


def speech_centres(st):
    t = np.arange(0, len(st["audio"]) / SR, HOP)
    keep = np.zeros(len(t), bool)
    for a, b in st["speech"]:
        keep |= (t >= a - 0.2) & (t <= b + 0.2)
    return t[keep]


def events(hit_times):
    """Joins windows in a row into one filler; returns the middle time of each run of MIN_WINDOWS+."""
    found, run = [], []
    for time in hit_times:
        if run and time - run[-1] > HOP * 1.5:
            if len(run) >= MIN_WINDOWS:
                found.append(float(np.mean(run)))
            run = []
        run.append(time)
    if len(run) >= MIN_WINDOWS:
        found.append(float(np.mean(run)))
    return np.array(found)


def detect(model, st):
    t = speech_centres(st)
    x = torch.from_numpy(windows(st["audio"], t)).to(dev)
    model.eval()
    with torch.no_grad():
        p = torch.cat([torch.softmax(model(x[i : i + 512]), 1) for i in range(0, len(x), 512)]).cpu().numpy()
    return events(t[p[:, FILLERS].sum(1) > THRESHOLD])


def matched(found, truth):
    """How many found fillers pair up, one to one, with a CrisperWhisper filler within MATCH_S."""
    used, hits = set(), 0
    for f in found:
        gaps = np.abs(np.asarray(truth) - f)
        for j in np.argsort(gaps):
            if gaps[j] > MATCH_S:
                break
            if j not in used:
                used.add(j)
                hits += 1
                break
    return hits


def evaluate(model_for):
    """model_for(code) gives the model used for that student."""
    rows = []
    for c, st in students.items():
        found = detect(model_for(c), st)
        rows.append(
            {
                "student": c,
                "minutes": round(len(st["audio"]) / SR / 60, 1),
                "crisperwhisper": len(st["fillers"]),
                "detector": len(found),
                "matched": matched(found, st["fillers"]),
            }
        )
    table = pd.DataFrame(rows)
    p = table.matched.sum() / max(table.detector.sum(), 1)
    r = table.matched.sum() / max(table.crisperwhisper.sum(), 1)
    summary = {
        "precision": p,
        "recall": r,
        "f1": 2 * p * r / max(p + r, 1e-9),
        "count_correlation": np.corrcoef(table.crisperwhisper, table.detector)[0, 1],
    }
    return table, summary

In [ ]:
# 4. The part 1 model as it is (it has never heard these students)
table_v1, summary_v1 = evaluate(lambda c: v1)
print(table_v1.to_string(index=False))
print("\nAll students:", {k: round(float(v), 3) for k, v in summary_v1.items()})

In [ ]:
# 5. Teach it the students' voices; each student is tested by a model that never heard them
EPOCHS, LR = 5, 1e-4


def clips(st):
    """Each filler (and two copies shifted by 0.15 s) as uh or um; other words, away from fillers, as Words."""
    centres, labels = [], []
    for time, tag in zip(st["fillers"], st["filler_tags"]):
        for shift in (-0.15, 0.0, 0.15):
            centres.append(time + shift)
            labels.append(LABELS.index("Um" if tag == "[UM]" else "Uh"))
    for time in st["words"]:
        if len(st["fillers"]) == 0 or np.abs(st["fillers"] - time).min() > 0.5:
            centres.append(time)
            labels.append(LABELS.index("Words"))
    return windows(st["audio"], centres), np.array(labels)


data = {c: clips(st) for c, st in students.items()}


def adapt(train_codes):
    x = torch.from_numpy(np.concatenate([data[c][0] for c in train_codes])).to(dev)
    y = torch.from_numpy(np.concatenate([data[c][1] for c in train_codes])).to(dev)
    model = copy.deepcopy(v1).train()  # starts from the part 1 model
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    for _ in range(EPOCHS):
        order = torch.randperm(len(y), device=dev)
        for i in range(0, len(order), 64):
            idx = order[i : i + 64]
            loss = nn.functional.cross_entropy(model(x[idx]), y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model.eval()


held_out = {c: adapt([o for o in codes if o != c]) for c in codes}
table_v2, summary_v2 = evaluate(lambda c: held_out[c])
print(table_v2.to_string(index=False))
print("\nAll students:", {k: round(float(v), 3) for k, v in summary_v2.items()})

In [ ]:
# 6. Compare, then train the final model on all students and save it to Drive
compare = pd.DataFrame({"part 1 model": summary_v1, "adapted (student unseen)": summary_v2}).astype(float).round(3)
print(compare)

v2 = adapt(codes)
ckpt2 = DRIVE / "filler_cnn_v2.pt"
torch.save({**ckpt, "state_dict": v2.state_dict(), "adapted_on": codes}, ckpt2)
results = {
    "model": ckpt2.name,
    "sha256": hashlib.sha256(ckpt2.read_bytes()).hexdigest(),
    "trained_at": datetime.datetime.now(ZoneInfo("Asia/Colombo")).isoformat(timespec="seconds"),
    "starts_from": "filler_cnn_v1.pt",
    "reference": "CrisperWhisper [UM] and [UH] words in each *_student.words.json",
    "students": codes,
    "settings": {
        "hop_s": HOP, "threshold": THRESHOLD, "min_windows": MIN_WINDOWS, "match_s": MATCH_S,
        "target_rms": TARGET_RMS, "silence_top_db": 40, "epochs": EPOCHS, "lr": LR, "seed": 0,
    },
    "part1_model": {"summary": summary_v1, "per_student": table_v1.to_dict("records")},
    "adapted_leave_one_student_out": {"summary": summary_v2, "per_student": table_v2.to_dict("records")},
    "torch": torch.__version__,
}
(DRIVE / "filler_cnn_v2_results.json").write_text(json.dumps(results, indent=2, default=float))
drive.flush_and_unmount()  # makes sure Drive has both files before you close the tab
print("\nSaved", ckpt2.name, results["sha256"])

**Done? Keep the log for your supervisors before closing anything:**
1. Press Ctrl+S.
2. File, Download, Download .ipynb, and keep it next to part 1 in
   `Research Data/Model ipynb`.

**Send the printed results of cells 4, 5 and 6 to Claude.** Then Runtime, Disconnect and delete
runtime: this removes the recordings from Colab.

How to read it: `crisperwhisper` is how many fillers CrisperWhisper marked, `detector` how many
the model found, `matched` how many of those are the same filler. CrisperWhisper is the reference
here, but it is not perfect either, so some "extra" detector fillers may be real ones it missed.